# GSE140901 — HCC Immunotherapy Response (PanCancer Immune Panel)

**Dataset**: [GSE140901](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE140901)
**Panel**: Nanostring nCounter PanCancer Immune Profiling v1.1 (730 endogenous + 40 housekeeping genes)
**Samples**: 24 hepatocellular carcinoma (HCC) tumors from patients treated with anti-PD-1/PD-L1 checkpoint inhibitors
**Design**: 13 responders (clinical benefit) vs 11 non-responders
**Source**: Ou et al.

This vignette uses ncountr to identify immune transcriptomic markers associated with immunotherapy response in HCC.

## 1. Download and Parse

In [ ]:
from ncountr.io.geo import fetch_geo
import ncountr
import pandas as pd

rcc_dir = fetch_geo("GSE140901", output_dir="data")

In [ ]:
# Clinical metadata from GEO sample pages
# clinical_benefit_response: Yes = responder (PR or durable SD), No = non-responder (PD or short SD)
CLINICAL = {
    "GSM4190059": ("S01", "NonResponder", "PD"),  "GSM4190060": ("S02", "NonResponder", "SD"),
    "GSM4190061": ("S03", "NonResponder", "PD"),  "GSM4190062": ("S06", "NonResponder", "PD"),
    "GSM4190063": ("S08", "Responder",    "PR"),  "GSM4190064": ("S10", "Responder",    "SD"),
    "GSM4190065": ("S11", "Responder",    "PR"),  "GSM4190066": ("S12", "Responder",    "SD"),
    "GSM4190067": ("S14", "NonResponder", "PD"),  "GSM4190068": ("S16", "Responder",    "SD"),
    "GSM4190069": ("S17", "NonResponder", "PD"),  "GSM4190070": ("S18", "Responder",    "PR"),
    "GSM4190071": ("S19", "Responder",    "PR"),  "GSM4190072": ("S20", "Responder",    "SD"),
    "GSM4190073": ("S21", "Responder",    "SD"),  "GSM4190074": ("S22", "Responder",    "PR"),
    "GSM4190075": ("S23", "NonResponder", "PD"),  "GSM4190076": ("S24", "NonResponder", "SD"),
    "GSM4190077": ("S25", "Responder",    "SD"),  "GSM4190078": ("S26", "Responder",    "SD"),
    "GSM4190079": ("S27", "NonResponder", "PD"),  "GSM4190080": ("S29", "NonResponder", "SD"),
    "GSM4190081": ("S30", "Responder",    "PR"),  "GSM4190082": ("S31", "NonResponder", "PD"),
}

exp = ncountr.read_rcc(rcc_dir, sample_id_pattern=r"(GSM\d+)", sample_id_from="filename")

# Rename GSM accessions to HCC sample IDs
rename = {gsm: info[0] for gsm, info in CLINICAL.items()}
exp.raw_counts = exp.raw_counts.rename(columns=rename)
exp.pos_counts = exp.pos_counts.rename(columns=rename)
exp.neg_counts = exp.neg_counts.rename(columns=rename)
exp.hk_counts = exp.hk_counts.rename(columns=rename)
exp.lane_info.index = [rename.get(s, s) for s in exp.lane_info.index]

meta = {info[0]: {"response": info[1], "best_response": info[2]} for info in CLINICAL.values()}
exp.sample_meta = pd.DataFrame.from_dict(meta, orient="index").rename_axis("sample")

print(exp)
print(f"\n{exp.sample_meta['response'].value_counts()}")

## 2. QC and Normalization

In [ ]:
qc_results = ncountr.qc(exp)
print(f"FOV pass: {qc_results['FovPass'].sum()}/{len(qc_results)}")
print(f"PosCtrl pass: {qc_results['PosCtrlPass'].sum()}/{len(qc_results)}")

from ncountr.plotting.qc_plots import plot_qc
fig = plot_qc(exp)
fig.tight_layout()

In [ ]:
ncountr.normalize(exp, method="pos_hk")
print(f"Normalized: {exp.normalized.shape[0]} genes x {exp.normalized.shape[1]} samples")

## 3. Differential Expression — Responder vs Non-Responder

In [ ]:
resp = [s for s in exp.samples if meta[s]["response"] == "Responder"]
nonresp = [s for s in exp.samples if meta[s]["response"] == "NonResponder"]

de_results = ncountr.de(exp, group_a=resp, group_b=nonresp, test="mannwhitneyu")

n_sig = (de_results["padj"] < 0.05).sum()
print(f"Responder ({len(resp)}) vs NonResponder ({len(nonresp)})")
print(f"Significant (padj < 0.05): {n_sig}")
print(f"\nTop 15 genes by p-value:")
de_results.sort_values("pvalue").head(15)[["gene", "log2FC", "pvalue", "padj"]]

In [ ]:
from ncountr.plotting.de_plots import plot_volcano

fig = plot_volcano(
    de_results,
    highlight_genes=ncountr.get_gene_set("IFN_JAKSTAT"),
    highlight_label="IFN/JAK-STAT",
    title="ICI Responder vs Non-Responder\n(GSE140901, PanCancer Immune panel)",
)

## 4. Gene Set Scoring and Heatmap

In [ ]:
scores = ncountr.score_gene_set(exp, gene_set="IFN_JAKSTAT")

score_df = pd.DataFrame({
    "sample": exp.samples,
    "IFN_JAKSTAT_score": [scores[s] for s in exp.samples],
    "response": [meta[s]["response"] for s in exp.samples],
})

from ncountr.plotting.pathway_plots import plot_pathway_scores
fig = plot_pathway_scores(
    score_df,
    score_column="IFN_JAKSTAT_score",
    group_column="response",
    title="IFN/JAK-STAT Pathway Score by Response\n(GSE140901)",
)

In [ ]:
# Heatmap of top DE genes
top_genes = list(de_results.sort_values("pvalue").head(25)["gene"])

# Order samples: responders then non-responders
ordered = sorted(exp.samples, key=lambda s: (0 if meta[s]["response"] == "Responder" else 1, s))

from ncountr.plotting.heatmaps import plot_heatmap
fig = plot_heatmap(
    exp,
    genes=top_genes,
    samples=ordered,
    z_score=True,
    title="Top 25 DE genes: Responder vs Non-Responder (z-scored)",
)

## 5. Export to AnnData

In [ ]:
adata = ncountr.to_anndata(exp)
print(adata)

## Summary

| Metric | Value |
|---|---|
| Samples | 24 (13 responders, 11 non-responders) |
| Genes tested | 730 |
| QC pass | 24/24 |
| Significant (FDR < 0.05) | 0 |
| Top hits (nominal) | C1QA, CYBB, HLA-DPA1, CD244, CXCL16, PDCD1LG2, LAG3 |

**Key observations:**
- No genes reach FDR significance with n=24, but the top nominally DE genes are highly biologically relevant
- Responders show higher expression of immune checkpoint markers (LAG3, CD244/2B4, PDCD1LG2/PD-L2) — consistent with "hot" tumors that respond to checkpoint blockade
- Antigen presentation genes (HLA-DPA1, HLA-DPB1) and myeloid markers (C1QA, CYBB) also higher in responders
- IFN/JAK-STAT pathway scores trend higher in responders (mean 0.09 vs -0.11)